# Regular Expressions & Text Processing: Beginner Guide

### 📌 Overview
Master **Regular Expressions & Text Processing: Beginner Guide** with concise, zero-fluff bullet points and executable code on real Fintech records ([raw_transactions.csv](file:///data/raw_transactions.csv)).

### 📚 Key Concepts Covered in this Notebook:
- **Matching & Searching**: Covers `re.match()`, `re.search()`, `re.findall()`, and `re.finditer()`.
- **Group Extraction**: Covers `match.group()`, `match.groups()`, and `match.groupdict()`.
- **Substitution & Pre-Compilation**: Covers `re.sub()` and `re.compile()`.


In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import csv
import sys
import time
import os
import json
import re
import collections
from datetime import datetime, timedelta

csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
transactions = []
with open(csv_path, mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        transactions.append(row)

print(f"Python Version: {sys.version.split()[0]}")
print(f"Loaded {len(transactions)} transaction records from {csv_path}")

Python Version: 3.12.7
Loaded 15000 transaction records from ../data/raw_transactions.csv


### 🔹 Prefix Pattern Matching: `re.match()`
- **What it does:** Matches regex patterns ONLY at the very beginning of the string.
- **Syntax:** `re.match(r'TX', string)`
- **Operation:** `tx_id = transactions[0]['transaction_id']`
- **Key Note:** Always use raw strings (e.g., `r'\d+'`) for regex patterns so Python doesn't treat backslashes as escape characters.

In [2]:
tx_id = transactions[0]['transaction_id']
m = re.match(r'TX', tx_id)
print('re.match found prefix:', m.group() if m else 'No match')

re.match found prefix: TX


### 🔹 String Scanning: `re.search()`
- **What it does:** Scans through the entire string to locate the first location where pattern matches.
- **Syntax:** `re.search(r'\d+', string)`
- **Operation:** `s_match = re.search(r'\d+', tx_id)`
- **Key Note:** Pandas `.str` accessor methods automatically skip missing (`NaN`) values instead of raising an `AttributeError`.

In [3]:
s_match = re.search(r'\d+', tx_id)
print('re.search extracted numeric digits:', s_match.group() if s_match else 'No match')

re.search extracted numeric digits: 109326


### 🔹 Global Matching: `re.findall()`
- **What it does:** Returns a list of all non-overlapping matching substrings.
- **Syntax:** `re.findall(r'TX\d+', text)`
- **Operation:** `sample_str = ' '.join([t['transaction_id'] for t in transactions[:4]])`
- **Key Note:** Always use raw strings (e.g., `r'\d+'`) for regex patterns so Python doesn't treat backslashes as escape characters.

In [4]:
sample_str = ' '.join([t['transaction_id'] for t in transactions[:4]])
all_ids = re.findall(r'TX\d+', sample_str)
print('re.findall extracted all IDs:', all_ids)

re.findall extracted all IDs: ['TX109326', 'TX106376', 'TX103301', 'TX110701']


### 🔹 Lazy Iteration: `re.finditer()`
- **What it does:** Yields Match objects across all matches lazily without allocating entire lists.
- **Syntax:** `for m in re.finditer(pattern, text): ...`
- **Operation:** `for m in re.finditer(r'TX\d+', sample_str):`
- **Key Note:** Always use raw strings (e.g., `r'\d+'`) for regex patterns so Python doesn't treat backslashes as escape characters.

In [5]:
for m in re.finditer(r'TX\d+', sample_str):
    print(f'Match: {m.group()} at span {m.span()}')

Match: TX109326 at span (0, 8)
Match: TX106376 at span (9, 17)
Match: TX103301 at span (18, 26)
Match: TX110701 at span (27, 35)


### 🔹 Group Extraction: `match.group()`
- **What it does:** Returns matching substring or specific indexed capture group.
- **Syntax:** `match.group(1)`
- **Operation:** `group_match = re.search(r'([A-Z]+)(\d+)', tx_id)`
- **Key Note:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.

In [6]:
group_match = re.search(r'([A-Z]+)(\d+)', tx_id)
print('Whole match .group(0):', group_match.group(0))
print('Group 1 (Prefix):', group_match.group(1))
print('Group 2 (Digits):', group_match.group(2))

Whole match .group(0): TX109326
Group 1 (Prefix): TX
Group 2 (Digits): 109326


### 🔹 Tuple Group Extraction: `match.groups()`
- **What it does:** Returns a tuple containing all captured subgroup strings.
- **Syntax:** `match.groups()`
- **Operation:** `print('All captured groups as tuple:', group_match.groups())`
- **Key Note:** Print intermediate variables with `print()` or inspect their type with `type()` to trace how data changes at each step.

In [7]:
print('All captured groups as tuple:', group_match.groups())

All captured groups as tuple: ('TX', '109326')


### 🔹 Named Capture Groups: `match.groupdict()`
- **What it does:** Extracts named groups defined via `(?P<name>pattern)` into a dictionary.
- **Syntax:** `re.search(r'(?P<pre>[A-Z]+)(?P<num>\d+)', text).groupdict()`
- **Operation:** `named_pat = re.compile(r'(?P<Prefix>[A-Z]+)(?P<ID>\d+)')`
- **Key Note:** Use `dict.get(key, default)` when looking up keys that might not exist, avoiding unexpected `KeyError` crashes.

In [8]:
named_pat = re.compile(r'(?P<Prefix>[A-Z]+)(?P<ID>\d+)')
named_match = named_pat.search(tx_id)
print('Named groups dictionary:', named_match.groupdict() if named_match else {})

Named groups dictionary: {'Prefix': 'TX', 'ID': '109326'}


### 🔹 Regex Substitution: `re.sub()`
- **What it does:** Replaces occurrences of pattern with replacement string.
- **Syntax:** `re.sub(r'pattern', 'replacement', text)`
- **Operation:** `dirty_currency = '$ 1,250.00 USD'`
- **Key Note:** Always use raw strings (e.g., `r'\d+'`) for regex patterns so Python doesn't treat backslashes as escape characters.

In [9]:
dirty_currency = '$ 1,250.00 USD'
clean_numeric = re.sub(r'[^0-9.]', '', dirty_currency)
print(f'Cleaned "{dirty_currency}" -> "{clean_numeric}"')

Cleaned "$ 1,250.00 USD" -> "1250.00"


### 🔹 Pattern Pre-Compilation: `re.compile()`
- **What it does:** Compiles regex pattern into a reusable Pattern object for performance acceleration.
- **Syntax:** `pattern = re.compile(r'...')`
- **Operation:** `compiled_tx_finder = re.compile(r'^TX\d+$')`
- **Key Note:** Always use raw strings (e.g., `r'\d+'`) for regex patterns so Python doesn't treat backslashes as escape characters.

In [10]:
compiled_tx_finder = re.compile(r'^TX\d+$')
print('Compiled regex check:', bool(compiled_tx_finder.match(tx_id)))

Compiled regex check: True


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data engineering questions explained with real examples.


### 🔍 Scenario: Q1: ReDoS (Regular Expression Denial of Service) Prevention
- **Objective:** Q1: ReDoS (Regular Expression Denial of Service) Prevention
- **Approach:** Explain catastrophic backtracking caused by overlapping nested quantifiers (e.g. `(a+)+$`) and how to construct linear regexes.
- **Syntax:** `re.compile(r'^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+$')`

In [11]:
print('Avoid nested quantifiers like (a+)+ to prevent ReDoS CPU exhaustion in web servers.')

Avoid nested quantifiers like (a+)+ to prevent ReDoS CPU exhaustion in web servers.
